# Genius API Sample Checks
Use this notebook to quickly search Genius and fetch lyrics for manual spot checks.

In [2]:
import os
import lyricsgenius

GENIUS_ACCESS_TOKEN = os.getenv("GENIUS_ACCESS_TOKEN", "")
if not GENIUS_ACCESS_TOKEN:
    raise ValueError("Set GENIUS_ACCESS_TOKEN in your environment or directly in this cell.")

genius = lyricsgenius.Genius(
    GENIUS_ACCESS_TOKEN,
    timeout=15,
    retries=3,
    remove_section_headers=True,
    skip_non_songs=True,
 )
genius.verbose = False

In [3]:
def fetch_lyrics_genius(client, title: str, artist: str) -> str:
    try:
        hit = client.search_song(title=title, artist=artist)
        if hit and hit.lyrics:
            return hit.lyrics.strip()
    except Exception as e:
        print(f"[warn] {title!r} by {artist!r}: {e}")
    return ""

In [ ]:
# STEP 1: Search Title using spotify_uri from titles.csv
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent  # go up from notebooks to project root

spotify_uri = "1OHj2WRXOR9XdCV6PuptXv"
titles_path = PROJECT_ROOT / "data" / "processed" / "titles.csv"
# DATABRICKS PATH
# titles_path = "/Volumes/songs_db/default/storage/titles.csv"

titles_df = pd.read_csv(titles_path)
matching_row = titles_df[titles_df["spotify_uri"] == spotify_uri]
song_title = matching_row["title"].iloc[0] if not matching_row.empty else None
song_artist = matching_row["artist"].iloc[0] if not matching_row.empty else None
print(f"\nSpotify URI: {spotify_uri}")
print(f"Title: {song_title}")
print(f"Artist: {song_artist}")


Spotify URI: 1OHj2WRXOR9XdCV6PuptXv
Title: Amarte más no pude - En vivo
Artist: Luister La Voz


In [ ]:
# STEP 1: Search Lyrics using Title
song_title = "(When You Gonna) Give It Up to Me - Radio Version"
song_artist = ""

In [6]:
# STEP 2: Fetch lyrics using Genius API
lyrics = fetch_lyrics_genius(genius, song_title, song_artist)

print("\nLyrics snippet:\n")
print((lyrics or "<no lyrics found>")[:1200])


Lyrics snippet:

Yeah, Ideal
Y-2-Kay Gee and R.L. collabo'
Uh-oh
Make 'em, make 'em, make 'em dance to this
Uh, yeah, Ideal, come on
Make 'em, make 'em, make 'em dance to this
Let's go

Whatever you wanna do, uh-uh
Whether it's at the crib or club
It's all up to you, what-what
Whatever you wanna do
So you wanna groove, uh-uh
Show me how you move, uh-huh
What you gonna do, what-what
'Cause I wanna rock with you
Whatever you wanna do

Tonight, you're with me
I'm talkin' VIP
Cristal, Moet, Dom P tonight, oh
Nice suite at the Double Tree
You and me in between the sheets
It's whatever you like, baby, tell me

I like when you move up close to me
Sexy grindin', holdin' me
Real close 'cause you're supposed to be, baby
I like it when you put that thing on me
Pushin' it up against the jeans
Makin' it harder in between, baby

Whatever you wanna do, uh-uh
Whether it's at the crib or club (Yeah, yeah, yeah)
It's all up to you (What-what), what-what
Whatever you wanna do
So you wanna groove (Oh), u

In [7]:
# STEP 2: Get spotify_uri from titles.csv by title and artist
# If artist is empty, search only by title

from pathlib import Path
import pandas as pd

song_title_tmp = '' # for manual searching if we want to try a different title

if song_title_tmp:
    song_title = song_title_tmp

if song_artist != "":
    matching_row = titles_df[
        titles_df["title"].str.contains(song_title, case=False, na=False) &
        titles_df["artist"].str.contains(song_artist, case=False, na=False)
    ]
else:
    matching_row = titles_df[
        titles_df["title"].str.contains(song_title, case=False, na=False)
    ]

result = titles_df.loc[titles_df["title"] == song_title, ["title", "spotify_uri", "artist"]]
print(result)


                                                 title  \
251  (When You Gonna) Give It Up to Me - Radio Version   

                spotify_uri                   artist  
251  5nEdwtSv0qqeE0l4o1lR4q  Sean Paul, Keyshia Cole  


/var/folders/z5/cwz1h7pd4kz7mtr5m_vdxj080000gn/T/ipykernel_9002/1238264472.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  titles_df["title"].str.contains(song_title, case=False, na=False)


In [ ]:
# STEP 3: Update titles.csv with song_title and song_artist for the row with the matching spotify_uri (if found)
if 'song_uri' in dir() and song_uri:
    titles_df.loc[titles_df["spotify_uri"] == song_uri, ["title", "artist"]] = [song_title, song_artist]
    titles_df.to_csv(Path("/Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/titles/2026/03/05/titles.csv"), index=False)
    print(f"Updated titles.csv with title '{song_title}' and artist '{song_artist}' for spotify_uri '{song_uri}'.")
else:
    print(f"song_uri not defined or empty. No updates made to titles.csv.")

Updated titles.csv with title 'Amarte más no pude - En vivo' and artist 'Luister La Voz' for spotify_uri '1OHj2WRXOR9XdCV6PuptXv'.


In [53]:
# STEP 4: Update lyrics.csv with lyrics for the matching spotify_uri
lyrics_df = pd.read_csv(Path("/Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics.csv"))
if song_uri and lyrics:
    lyrics_df.loc[lyrics_df["spotify_uri"] == song_uri, "lyrics"] = lyrics
    lyrics_df.to_csv(Path("/Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics.csv"), index=False)
    print(f"Updated lyrics.csv with lyrics for spotify_uri '{song_uri}'.")
else:
    print(f"No matching spotify_uri or lyrics found for title '{song_title}' and artist '{song_artist}'. No updates made to lyrics.csv.")

Updated lyrics.csv with lyrics for spotify_uri '1OHj2WRXOR9XdCV6PuptXv'.
